In [1]:
# ============================================
# Startup Cell: Mount Drive + Prepare Data
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"
CONTENT_DIR = "/content"

# -------------------------------------------------
# Feature vector files
# -------------------------------------------------
INPUT_FILES = [
    "train_feature_vectors.csv",
    "validation_feature_vectors.csv",
    "test_feature_vectors.csv"
]

# -------------------------------------------------
# Copy files to /content
# -------------------------------------------------
print("Copying feature vector CSV files...")

for filename in INPUT_FILES:
    src = os.path.join(BASE_DRIVE_DIR, filename)
    dst = os.path.join(CONTENT_DIR, filename)

    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing source file: {src}")

    shutil.copy(src, dst)
    print(f"Copied: {filename}")

print("\nFeature vector CSVs copied.")

# -------------------------------------------------
# Verification
# -------------------------------------------------
print("\nVerification:")
all_ok = True

for filename in INPUT_FILES:
    path = os.path.join(CONTENT_DIR, filename)
    exists = os.path.exists(path)
    print(f"{filename:<40} {'OK' if exists else 'MISSING'}")
    all_ok = all_ok and exists

print("\nAll files present." if all_ok else "\nSome files are missing.")



Mounted at /content/drive
Copying feature vector CSV files...
Copied: train_feature_vectors.csv
Copied: validation_feature_vectors.csv
Copied: test_feature_vectors.csv

Feature vector CSVs copied.

Verification:
train_feature_vectors.csv                OK
validation_feature_vectors.csv           OK
test_feature_vectors.csv                 OK

All files present.


In [2]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
#
# Purpose:
# Prepare normalized, classifier-ready inputs from the train,
# validation, and test feature-vector CSV files.
#
# Inputs:
#   /content/train_feature_vectors.csv
#   /content/validation_feature_vectors.csv
#   /content/test_feature_vectors.csv
#
# Assumptions:
# - Each input CSV contains 4 metadata columns plus 26 DIP features
# - Metadata columns are:
#     filename
#     class_label
#     source_dataset
#     subset
# - class_label contains:
#     "ai"   for AI-generated images
#     "real" for real images
#
# Main steps:
# 1. Load train, validation, and test feature-vector CSV files
# 2. Separate metadata columns from numeric feature columns
# 3. Encode class labels for binary classification
# 4. Fit a feature scaler on the training set only
# 5. Apply the same scaler to the validation and test sets
# 6. Save normalized feature-vector files, scaler, and debug CSV
#    for downstream classifier training
#
# Scaler determination:
# - The scaler is computed using training data only
# - For each feature, the training-set mean and standard deviation
#   are used to define a standardization transform
# - The scaler stores one set of parameters (mean and standard
#   deviation) for each of the 26 features
# - Validation and test data are transformed using this same
#   training-set scaler without refitting
#
# Outputs:
# - Normalized feature-vector CSV files:
#     /content/train_feature_vectors_normalized.csv
#     /content/validation_feature_vectors_normalized.csv
#     /content/test_feature_vectors_normalized.csv
# - Human-readable scaler debug CSV:
#     /content/scaler_debug.csv
# - Saved scaler object (scaler.pkl), which contains the per-feature
#   scaling parameters used for normalization
#
# Notes:
# - Fitting the scaler on training data only prevents data leakage
# - Validation and test sets remain independent of training
# - Feature, label, and metadata separation is performed in memory
# - A readable scaler_debug.csv file is saved for debugging and inspection
# - No classifier training is performed in this notebook
# ============================================


In [3]:
# ============================================
# Cell 1: Imports and Base Paths
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler

BASE_DIR = Path("/content")
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = BASE_DIR / "train_feature_vectors.csv"
VALIDATION_CSV = BASE_DIR / "validation_feature_vectors.csv"
TEST_CSV = BASE_DIR / "test_feature_vectors.csv"

TRAIN_NORMALIZED_CSV = OUTPUT_DIR / "train_feature_vectors_normalized.csv"
VALIDATION_NORMALIZED_CSV = OUTPUT_DIR / "validation_feature_vectors_normalized.csv"
TEST_NORMALIZED_CSV = OUTPUT_DIR / "test_feature_vectors_normalized.csv"

SCALER_PATH = OUTPUT_DIR / "scaler.pkl"
SCALER_DEBUG_CSV = OUTPUT_DIR / "scaler_debug.csv"

print("TRAIN_CSV                 =", TRAIN_CSV)
print("VALIDATION_CSV            =", VALIDATION_CSV)
print("TEST_CSV                  =", TEST_CSV)
print("TRAIN_NORMALIZED_CSV      =", TRAIN_NORMALIZED_CSV)
print("VALIDATION_NORMALIZED_CSV =", VALIDATION_NORMALIZED_CSV)
print("TEST_NORMALIZED_CSV       =", TEST_NORMALIZED_CSV)
print("SCALER_PATH               =", SCALER_PATH)
print("SCALER_DEBUG_CSV          =", SCALER_DEBUG_CSV)


TRAIN_CSV                 = /content/train_feature_vectors.csv
VALIDATION_CSV            = /content/validation_feature_vectors.csv
TEST_CSV                  = /content/test_feature_vectors.csv
TRAIN_NORMALIZED_CSV      = /content/train_feature_vectors_normalized.csv
VALIDATION_NORMALIZED_CSV = /content/validation_feature_vectors_normalized.csv
TEST_NORMALIZED_CSV       = /content/test_feature_vectors_normalized.csv
SCALER_PATH               = /content/scaler.pkl
SCALER_DEBUG_CSV          = /content/scaler_debug.csv


In [4]:
# ============================================
# Cell 2: Define Metadata, Label, and Feature Columns
# ============================================

METADATA_COLS = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

LABEL_COL = "class_label"

EXPECTED_FEATURE_COUNT = 26

print("METADATA_COLS          =", METADATA_COLS)
print("LABEL_COL              =", LABEL_COL)
print("EXPECTED_FEATURE_COUNT =", EXPECTED_FEATURE_COUNT)



METADATA_COLS          = ['filename', 'class_label', 'source_dataset', 'subset']
LABEL_COL              = class_label
EXPECTED_FEATURE_COUNT = 26


In [5]:
# ============================================
# Cell 3: Load Feature Vector CSVs and Validate Schema
# ============================================

# -------------------------------------------------
# Check files exist
# -------------------------------------------------
for path in [TRAIN_CSV, VALIDATION_CSV, TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("PASS: All input CSV files found")

# -------------------------------------------------
# Load CSVs
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_CSV)
df_validation = pd.read_csv(VALIDATION_CSV)
df_test = pd.read_csv(TEST_CSV)

print("\nLoaded DataFrames:")
print("Train shape      :", df_train.shape)
print("Validation shape :", df_validation.shape)
print("Test shape       :", df_test.shape)

# -------------------------------------------------
# Validate required columns
# -------------------------------------------------
required_cols = METADATA_COLS

for name, df in [("train", df_train), ("validation", df_validation), ("test", df_test)]:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name} CSV missing required columns: {missing}")

print("\nPASS: Required metadata and label columns present")

# -------------------------------------------------
# Identify feature columns
# -------------------------------------------------
feature_cols = [
    col for col in df_train.columns
    if col not in METADATA_COLS + [LABEL_COL]
]

print("\nDetected feature columns:", len(feature_cols))

if len(feature_cols) != EXPECTED_FEATURE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features, found {len(feature_cols)}"
    )

# -------------------------------------------------
# Validate feature consistency across splits
# -------------------------------------------------
for name, df in [("validation", df_validation), ("test", df_test)]:
    cols = [
        col for col in df.columns
        if col not in METADATA_COLS + [LABEL_COL]
    ]
    if cols != feature_cols:
        raise ValueError(f"Feature column mismatch between train and {name}")

print("PASS: Feature columns consistent across all splits")

# -------------------------------------------------
# Validate subset column values
# -------------------------------------------------
for name, df in [("train", df_train), ("validation", df_validation), ("test", df_test)]:
    unique_vals = sorted(df["subset"].dropna().unique().tolist())
    if unique_vals != [name]:
        raise ValueError(f"{name} subset mismatch: expected [{name}], got {unique_vals}")

print("PASS: Subset column values match expected splits")



PASS: All input CSV files found

Loaded DataFrames:
Train shape      : (8400, 30)
Validation shape : (1800, 30)
Test shape       : (1800, 30)

PASS: Required metadata and label columns present

Detected feature columns: 26
PASS: Feature columns consistent across all splits
PASS: Subset column values match expected splits


In [6]:
# ============================================
# Cell 4: Separate Metadata, Labels, and Features
# ============================================

# -------------------------------------------------
# Create metadata DataFrames
# -------------------------------------------------
meta_train = df_train[METADATA_COLS].copy()
meta_validation = df_validation[METADATA_COLS].copy()
meta_test = df_test[METADATA_COLS].copy()

# -------------------------------------------------
# Create label vectors
# -------------------------------------------------
y_train = df_train[LABEL_COL].copy()
y_validation = df_validation[LABEL_COL].copy()
y_test = df_test[LABEL_COL].copy()

# -------------------------------------------------
# Create feature matrices
# -------------------------------------------------
X_train = df_train[feature_cols].copy()
X_validation = df_validation[feature_cols].copy()
X_test = df_test[feature_cols].copy()

# -------------------------------------------------
# Display shapes
# -------------------------------------------------
print("Metadata shapes:")
print("meta_train      :", meta_train.shape)
print("meta_validation :", meta_validation.shape)
print("meta_test       :", meta_test.shape)

print("\nLabel shapes:")
print("y_train         :", y_train.shape)
print("y_validation    :", y_validation.shape)
print("y_test          :", y_test.shape)

print("\nFeature shapes:")
print("X_train         :", X_train.shape)
print("X_validation    :", X_validation.shape)
print("X_test          :", X_test.shape)

# -------------------------------------------------
# Quick validation
# -------------------------------------------------
assert X_train.shape[1] == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} training features, got {X_train.shape[1]}"
assert X_validation.shape[1] == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} validation features, got {X_validation.shape[1]}"
assert X_test.shape[1] == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} test features, got {X_test.shape[1]}"

assert len(X_train) == len(y_train) == len(meta_train), "Train split length mismatch"
assert len(X_validation) == len(y_validation) == len(meta_validation), "Validation split length mismatch"
assert len(X_test) == len(y_test) == len(meta_test), "Test split length mismatch"

print("\nPASS: Metadata, labels, and features separated successfully")



Metadata shapes:
meta_train      : (8400, 4)
meta_validation : (1800, 4)
meta_test       : (1800, 4)

Label shapes:
y_train         : (8400,)
y_validation    : (1800,)
y_test          : (1800,)

Feature shapes:
X_train         : (8400, 26)
X_validation    : (1800, 26)
X_test          : (1800, 26)

PASS: Metadata, labels, and features separated successfully


In [7]:
# ============================================
# Cell 5: Encode Class Labels
# ============================================

# -------------------------------------------------
# Define label mapping
# -------------------------------------------------
LABEL_MAP = {
    "real": 0,
    "ai": 1
}

# -------------------------------------------------
# Encode labels
# -------------------------------------------------
y_train_encoded = y_train.map(LABEL_MAP)
y_validation_encoded = y_validation.map(LABEL_MAP)
y_test_encoded = y_test.map(LABEL_MAP)

# -------------------------------------------------
# Validate encoding
# -------------------------------------------------
for name, y_raw, y_enc in [
    ("train", y_train, y_train_encoded),
    ("validation", y_validation, y_validation_encoded),
    ("test", y_test, y_test_encoded),
]:
    if y_enc.isnull().any():
        bad_labels = sorted(y_raw[y_enc.isnull()].unique().tolist())
        raise ValueError(f"Unmapped labels found in {name}: {bad_labels}")

print("PASS: Labels encoded successfully")

print("\nLabel mapping:")
print(LABEL_MAP)

print("\nEncoded label counts:")
print("y_train_encoded:")
print(y_train_encoded.value_counts().sort_index())

print("\ny_validation_encoded:")
print(y_validation_encoded.value_counts().sort_index())

print("\ny_test_encoded:")
print(y_test_encoded.value_counts().sort_index())

# -------------------------------------------------
# Convert to NumPy arrays for downstream use
# -------------------------------------------------
y_train_encoded = y_train_encoded.to_numpy(dtype=np.int64)
y_validation_encoded = y_validation_encoded.to_numpy(dtype=np.int64)
y_test_encoded = y_test_encoded.to_numpy(dtype=np.int64)

print("\nEncoded label array shapes:")
print("y_train_encoded      :", y_train_encoded.shape)
print("y_validation_encoded :", y_validation_encoded.shape)
print("y_test_encoded       :", y_test_encoded.shape)



PASS: Labels encoded successfully

Label mapping:
{'real': 0, 'ai': 1}

Encoded label counts:
y_train_encoded:
class_label
0    4200
1    4200
Name: count, dtype: int64

y_validation_encoded:
class_label
0    900
1    900
Name: count, dtype: int64

y_test_encoded:
class_label
0    900
1    900
Name: count, dtype: int64

Encoded label array shapes:
y_train_encoded      : (8400,)
y_validation_encoded : (1800,)
y_test_encoded       : (1800,)


In [8]:
# ============================================
# Cell 6: Fit Scaler on Training Data and Transform All Splits
# ============================================

# -------------------------------------------------
# Initialize scaler
# -------------------------------------------------
scaler = StandardScaler()

# -------------------------------------------------
# Fit on training features only
# -------------------------------------------------
X_train_scaled = scaler.fit_transform(X_train)

# -------------------------------------------------
# Apply same scaler to validation and test features
# -------------------------------------------------
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------
# Convert back to DataFrames for easier inspection/saving
# -------------------------------------------------
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_validation_scaled_df = pd.DataFrame(X_validation_scaled, columns=feature_cols, index=X_validation.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

# -------------------------------------------------
# Display shapes
# -------------------------------------------------
print("Scaled feature shapes:")
print("X_train_scaled      :", X_train_scaled_df.shape)
print("X_validation_scaled :", X_validation_scaled_df.shape)
print("X_test_scaled       :", X_test_scaled_df.shape)

# -------------------------------------------------
# Basic validation
# -------------------------------------------------
assert X_train_scaled_df.shape == X_train.shape, "Scaled train shape mismatch"
assert X_validation_scaled_df.shape == X_validation.shape, "Scaled validation shape mismatch"
assert X_test_scaled_df.shape == X_test.shape, "Scaled test shape mismatch"

print("\nPASS: Scaler fit on training data and applied to all splits")

# -------------------------------------------------
# Show scaler parameter dimensions
# -------------------------------------------------
print("\nScaler parameter shapes:")
print("scaler.mean_  :", scaler.mean_.shape)
print("scaler.scale_ :", scaler.scale_.shape)

assert len(scaler.mean_) == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} scaler means, got {len(scaler.mean_)}"
assert len(scaler.scale_) == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} scaler scales, got {len(scaler.scale_)}"

print("\nPASS: Scaler contains per-feature parameters for all 26 features")



Scaled feature shapes:
X_train_scaled      : (8400, 26)
X_validation_scaled : (1800, 26)
X_test_scaled       : (1800, 26)

PASS: Scaler fit on training data and applied to all splits

Scaler parameter shapes:
scaler.mean_  : (26,)
scaler.scale_ : (26,)

PASS: Scaler contains per-feature parameters for all 26 features


In [9]:
# ============================================
# Cell 7: Save Normalized Feature Vectors, Scaler, and Debug CSV
# ============================================

# -------------------------------------------------
# Reconstruct normalized DataFrames (metadata + label + scaled features)
# -------------------------------------------------
train_normalized_df = pd.concat(
    [meta_train.reset_index(drop=True), X_train_scaled_df.reset_index(drop=True)],
    axis=1
)

validation_normalized_df = pd.concat(
    [meta_validation.reset_index(drop=True), X_validation_scaled_df.reset_index(drop=True)],
    axis=1
)

test_normalized_df = pd.concat(
    [meta_test.reset_index(drop=True), X_test_scaled_df.reset_index(drop=True)],
    axis=1
)

# -------------------------------------------------
# Save normalized CSV files
# -------------------------------------------------
train_normalized_df.to_csv(TRAIN_NORMALIZED_CSV, index=False)
validation_normalized_df.to_csv(VALIDATION_NORMALIZED_CSV, index=False)
test_normalized_df.to_csv(TEST_NORMALIZED_CSV, index=False)

print("Saved normalized CSV files:")
print(" -", TRAIN_NORMALIZED_CSV)
print(" -", VALIDATION_NORMALIZED_CSV)
print(" -", TEST_NORMALIZED_CSV)

# -------------------------------------------------
# Save scaler object for later reuse
# -------------------------------------------------
joblib.dump(scaler, SCALER_PATH)

print("\nSaved scaler object:")
print(" -", SCALER_PATH)

# -------------------------------------------------
# Save human-readable scaler parameters for debugging
# -------------------------------------------------
scaler_debug_df = pd.DataFrame({
    "feature": feature_cols,
    "mean": scaler.mean_,
    "std": scaler.scale_,
    "variance": scaler.var_
})

scaler_debug_df.to_csv(SCALER_DEBUG_CSV, index=False)

print("\nSaved scaler debug CSV:")
print(" -", SCALER_DEBUG_CSV)

# -------------------------------------------------
# Final verification
# -------------------------------------------------
for path in [
    TRAIN_NORMALIZED_CSV,
    VALIDATION_NORMALIZED_CSV,
    TEST_NORMALIZED_CSV,
    SCALER_PATH,
    SCALER_DEBUG_CSV,
]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing output file: {path}")

print("\nPASS: All normalized files, scaler, and debug CSV saved successfully")

# -------------------------------------------------
# Display final shapes and sample scaler rows
# -------------------------------------------------
print("\nFinal output shapes:")
print("Train normalized      :", train_normalized_df.shape)
print("Validation normalized :", validation_normalized_df.shape)
print("Test normalized       :", test_normalized_df.shape)

print("\nFirst 10 scaler rows:")
print(scaler_debug_df.head(10).to_string(index=False))


Saved normalized CSV files:
 - /content/train_feature_vectors_normalized.csv
 - /content/validation_feature_vectors_normalized.csv
 - /content/test_feature_vectors_normalized.csv

Saved scaler object:
 - /content/scaler.pkl

Saved scaler debug CSV:
 - /content/scaler_debug.csv

PASS: All normalized files, scaler, and debug CSV saved successfully

Final output shapes:
Train normalized      : (8400, 30)
Validation normalized : (1800, 30)
Test normalized       : (1800, 30)

First 10 scaler rows:
            feature     mean      std  variance
      Mean Gradient 0.302928 0.156065  0.024356
       Std Gradient 0.395674 0.165097  0.027257
       Max Gradient 3.182113 0.698673  0.488144
   Gradient Entropy 3.698846 0.715032  0.511271
       Edge Density 0.117505 0.024271  0.000589
   Orientation Mean 0.042007 0.152125  0.023142
    Orientation Std 1.776834 0.093102  0.008668
Orientation Entropy 4.993168 0.216910  0.047050
     Global Entropy 7.234426 0.539492  0.291052
 Local Entropy Mean 2.

In [10]:
# ============================================
# Cell 8: Final Sanity Check
# ============================================

import numpy as np

# -------------------------------------------------
# Reload saved files
# -------------------------------------------------
df_train_norm = pd.read_csv(TRAIN_NORMALIZED_CSV)
df_validation_norm = pd.read_csv(VALIDATION_NORMALIZED_CSV)
df_test_norm = pd.read_csv(TEST_NORMALIZED_CSV)
df_scaler_debug = pd.read_csv(SCALER_DEBUG_CSV)

print("Reloaded normalized files and scaler debug CSV")

# -------------------------------------------------
# Identify feature columns again
# -------------------------------------------------
feature_cols_check = [
    col for col in df_train_norm.columns
    if col not in METADATA_COLS
]

# -------------------------------------------------
# Shape checks
# -------------------------------------------------
print("\nShape check:")
print("Train      :", df_train_norm.shape)
print("Validation :", df_validation_norm.shape)
print("Test       :", df_test_norm.shape)
print("Scaler CSV :", df_scaler_debug.shape)

assert df_scaler_debug.shape[0] == EXPECTED_FEATURE_COUNT, \
    f"Expected {EXPECTED_FEATURE_COUNT} rows in scaler debug CSV, got {df_scaler_debug.shape[0]}"

# -------------------------------------------------
# Missing values check
# -------------------------------------------------
print("\nMissing values check:")
print("Train      :", df_train_norm.isnull().sum().sum())
print("Validation :", df_validation_norm.isnull().sum().sum())
print("Test       :", df_test_norm.isnull().sum().sum())
print("Scaler CSV :", df_scaler_debug.isnull().sum().sum())

assert df_train_norm.isnull().sum().sum() == 0, "NaNs in train"
assert df_validation_norm.isnull().sum().sum() == 0, "NaNs in validation"
assert df_test_norm.isnull().sum().sum() == 0, "NaNs in test"
assert df_scaler_debug.isnull().sum().sum() == 0, "NaNs in scaler debug CSV"

# -------------------------------------------------
# Scaling check (train only)
# -------------------------------------------------
X_train_check = df_train_norm[feature_cols_check]

means = X_train_check.mean()
stds = X_train_check.std(ddof=0)

print("\nScaling check (train):")
print("Mean (first 5):")
print(means.head())

print("\nStd (first 5):")
print(stds.head())

# Use scaler debug CSV to handle zero-variance features correctly
zero_var_mask = df_scaler_debug["variance"].to_numpy() == 0
nonzero_var_mask = ~zero_var_mask

mean_ok = np.allclose(means.to_numpy(), 0, atol=1e-6)
if np.any(nonzero_var_mask):
    std_ok = np.allclose(stds.to_numpy()[nonzero_var_mask], 1, atol=1e-6)
else:
    std_ok = True

print("\nScaling status:")
print("Means close to 0                :", mean_ok)
print("Nonzero-variance stds close to 1:", std_ok)
print("Zero-variance feature count     :", int(np.sum(zero_var_mask)))

assert mean_ok, "Train features not centered at 0"
assert std_ok, "Nonzero-variance train features not scaled to unit variance"

if np.any(zero_var_mask):
    print("\nZero-variance features detected:")
    print(df_scaler_debug.loc[zero_var_mask, ["feature", "mean", "std", "variance"]].to_string(index=False))

# -------------------------------------------------
# Subset consistency
# -------------------------------------------------
print("\nSubset check:")
print(df_train_norm["subset"].unique())
print(df_validation_norm["subset"].unique())
print(df_test_norm["subset"].unique())

assert list(df_train_norm["subset"].unique()) == ["train"]
assert list(df_validation_norm["subset"].unique()) == ["validation"]
assert list(df_test_norm["subset"].unique()) == ["test"]

print("\nPASS: Final sanity check successful")


Reloaded normalized files and scaler debug CSV

Shape check:
Train      : (8400, 30)
Validation : (1800, 30)
Test       : (1800, 30)
Scaler CSV : (26, 4)

Missing values check:
Train      : 0
Validation : 0
Test       : 0
Scaler CSV : 0

Scaling check (train):
Mean (first 5):
Mean Gradient      -2.239478e-16
Std Gradient       -2.916186e-16
Max Gradient        3.049413e-16
Gradient Entropy    3.793791e-16
Edge Density       -2.453064e-17
dtype: float64

Std (first 5):
Mean Gradient       1.0
Std Gradient        1.0
Max Gradient        1.0
Gradient Entropy    1.0
Edge Density        1.0
dtype: float64

Scaling status:
Means close to 0                : True
Nonzero-variance stds close to 1: True
Zero-variance feature count     : 0

Subset check:
['train']
['validation']
['test']

PASS: Final sanity check successful
